In [13]:
import requests
import mysql.connector
from dotenv import load_dotenv
import os
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd

load_dotenv()

BASE_URL = os.getenv("BASE_URL")
driver_service = Service("C:\chromedriver\chromedriver-win64\chromedriver.exe")


In [20]:
def minutes_to_seconds(time_str):
    minutes, seconds = map(int, time_str.split(':'))
    return minutes * 60 + seconds

def get_season_stats(team, season_year, driver):
    url = f"{BASE_URL}teams/{team}/{season_year}/gamelog/"
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.ID, f"gamelog{season_year}"))
        )
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Find the game table
        tables = [soup.find('table', id=f"gamelog{season_year}"), soup.find('table', id=f"playoff_gamelog{season_year}")]
        stats = []
        win_percentage = 0
        tot_games = 0

        for table in tables:
            if not table:
                print("Game-by-game table not found.")
                continue

            # Process rows
            rows = table.find('tbody').find_all('tr')
            if not rows:
                print("No rows found in the table.")
                continue

            for row in rows:
                # Skip empty rows (e.g., placeholders for playoff games not yet played)
                print(row.find('td', {'data-stat': 'boxscore_word'}).text)
                if row.find('td', {'data-stat': 'boxscore_word'}).text == "preview" or row.find('td', {'data-stat': 'opp'}).text == "Bye Week":
                    continue
                if row.get('class') and 'thead' in row['class']:
                    continue
                # Extract basic stats
                game_week = row.find('th', {'data-stat': 'week_num'})
                overtime = row.find('td', {'data-stat': 'overtime'})
                outcome = row.find('td', {'data-stat': 'game_outcome'})
                tot_games += 1
                if outcome.text == 'W':
                    win_percentage += 1
                game_location = row.find('td', {'data-stat': 'game_location'})
                opponent = row.find('td', {'data-stat': 'opp'})
                points_scored = row.find('td', {'data-stat': 'pts_off'})
                points_allowed = row.find('td', {'data-stat': 'pts_def'})
                passes_completed = row.find('td', {'data-stat': 'pass_cmp'})
                passes_attempted = row.find('td', {'data-stat': 'pass_att'})
                passing_yards = row.find('td', {'data-stat': 'pass_yds'})
                passing_td = row.find('td', {'data-stat': 'pass_td'})
                interceptions = row.find('td', {'data-stat': 'pass_int'})
                times_sacked = row.find('td', {'data-stat': 'pass_sacked'})
                yards_sacked = row.find('td', {'data-stat': 'pass_sacked_yds'})
                yards_per_attempt = row.find('td', {'data-stat': 'pass_yds_per_att'})
                net_yards_per_attempt = row.find('td', {'data-stat': 'pass_net_yds_per_att'})
                completion_rate = row.find('td', {'data-stat': 'pass_cmp_perc'})
                passer_rating = row.find('td', {'data-stat': 'pass_rating'})
                rushing_attempts = row.find('td', {'data-stat': 'rush_att'})
                rushing_yards = row.find('td', {'data-stat': 'rush_yds'})
                rushing_yards_per_attempt = row.find('td', {'data-stat': 'rush_yds_per_att'})
                rushing_td = row.find('td', {'data-stat': 'rush_td'})
                field_goals_made = row.find('td', {'data-stat': 'fgm'})
                field_goals_attempted = row.find('td', {'data-stat': 'fga'})
                extra_points_made = row.find('td', {'data-stat': 'xpm'})
                extra_points_attempted = row.find('td', {'data-stat': 'xpa'})
                punts = row.find('td', {'data-stat': 'punt'})
                punting_yards = row.find('td', {'data-stat': 'punt_yds'})
                third_down_conversions = row.find('td', {'data-stat': 'third_down_success'})
                third_down_attempts = row.find('td', {'data-stat': 'third_down_att'})
                fourth_down_conversions = row.find('td', {'data-stat': 'fourth_down_success'})
                fourth_down_attempts = row.find('td', {'data-stat': 'fourth_down_att'})
                time_in_possession = row.find('td', {'data-stat': 'time_of_poss'})
                if game_week and points_scored:
                    stats.append({
                        "Game Week": int(game_week.text) if game_week else None,
                        "Win Percentage": win_percentage / tot_games,
                        "Overtime": 1 if overtime and overtime.text else (0 if overtime else None),
                        "Outcome": 1 if outcome and outcome.text == "W" else (0 if outcome else None),
                        "Home Game": 0 if game_location and game_location.text == "@" else (1 if game_location else None),
                        "Opponent": opponent.text.split(" ")[-1] if opponent else None,
                        "Opponent Abbreviation": opponent.find('a')['href'].split('/')[2] if opponent and opponent.find('a') else None,
                        "Points Scored": int(points_scored.text) if points_scored else None,
                        "Points Allowed": int(points_allowed.text) if points_allowed else None,
                        "Passes Completed": int(passes_completed.text) if passes_completed else None,
                        "Passes Attempted": int(passes_attempted.text) if passes_attempted else None,
                        "Passing Yards": int(passing_yards.text) if passing_yards else None,
                        "Passing TD": int(passing_td.text) if passing_td else None,
                        "Interceptions": int(interceptions.text) if interceptions else None,
                        "Times Sacked": int(times_sacked.text) if times_sacked else None,
                        "Yards Sacked": int(yards_sacked.text) if yards_sacked else None,
                        "Yards per Attempt": float(yards_per_attempt.text) if yards_per_attempt else None,
                        "Net Yards per Attempt": float(net_yards_per_attempt.text) if net_yards_per_attempt else None,
                        "Completion Rate": float(completion_rate.text) if completion_rate else None,
                        "Passer Rating": float(passer_rating.text) if passer_rating else None,
                        "Rushing Attempts": int(rushing_attempts.text) if rushing_attempts else None,
                        "Rushing Yards": int(rushing_yards.text) if rushing_yards else None,
                        "Rushing Yards per Attempt": float(rushing_yards_per_attempt.text) if rushing_yards_per_attempt else None,
                        "Rushing TD": int(rushing_td.text) if rushing_td else None,
                        "Field Goals Made": int(field_goals_made.text) if field_goals_made else None,
                        "Field Goals Attempted": int(field_goals_attempted.text) if field_goals_attempted else None,
                        "Extra Points Made": int(extra_points_made.text) if extra_points_made else None,
                        "Extra Points Attempted": int(extra_points_attempted.text) if extra_points_attempted else None,
                        "Punts": int(punts.text) if punts else None,
                        "Punting Yards": int(punting_yards.text) if punting_yards else None,
                        "Third Down Conversions": int(third_down_conversions.text) if third_down_conversions else None,
                        "Third Down Attempts": int(third_down_attempts.text) if third_down_attempts else None,
                        "Fourth Down Conversions": int(fourth_down_conversions.text) if fourth_down_conversions else None,
                        "Fourth Down Attempts": int(fourth_down_attempts.text) if fourth_down_attempts else None,
                        "Time of Possession in Seconds": minutes_to_seconds(time_in_possession.text) if time_in_possession and time_in_possession.text else None,
                    })

        return stats

    finally:
        driver.quit()


In [15]:
super_bowl_years = [str(year) for year in range(2024, 1966 - 1, -1)]
super_bowl_years.reverse()
print(super_bowl_years)

['1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']


In [16]:
url = f"{BASE_URL}/teams/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Find team abbreviations and names
teams = {}
for row in soup.find_all('th', {'data-stat': 'team_name'}):
    a = row.find('a')
    if a:
        team_name = a.text
        team_abbreviation = a['href'].split('/')[2]
        teams[team_abbreviation] = team_name.split(" ")[-1]
print(teams)

{'crd': 'Cardinals', 'atl': 'Falcons', 'rav': 'Ravens', 'buf': 'Bills', 'car': 'Panthers', 'chi': 'Bears', 'cin': 'Bengals', 'cle': 'Browns', 'dal': 'Cowboys', 'den': 'Broncos', 'det': 'Lions', 'gnb': 'Packers', 'htx': 'Texans', 'clt': 'Colts', 'jax': 'Jaguars', 'kan': 'Chiefs', 'rai': 'Raiders', 'sdg': 'Chargers', 'ram': 'Rams', 'mia': 'Dolphins', 'min': 'Vikings', 'nwe': 'Patriots', 'nor': 'Saints', 'nyg': 'Giants', 'nyj': 'Jets', 'phi': 'Eagles', 'pit': 'Steelers', 'sfo': '49ers', 'sea': 'Seahawks', 'tam': 'Buccaneers', 'oti': 'Titans', 'was': 'Commanders'}


In [17]:
db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    auth_plugin="caching_sha2_password"
)
cursor = db.cursor()


In [18]:
league_id = 1
season_ids = {}
team_name_ids = {}
team_abr_ids = {}

get_season_ids = "SELECT * FROM seasons"
get_team_ids = "SELECT * FROM teams"

cursor.execute(get_season_ids)

#print(cursor.fetchall())
for season in cursor.fetchall():
    season_ids[season[2]] = season[0]
print(season_ids)
cursor.execute(get_team_ids)
for team in cursor.fetchall():
    team_name_ids[team[2]] = team[0]
    team_abr_ids[team[3]] = team[0]
print(team_name_ids)
print(team_abr_ids)

#print(cursor.fetchall())

{2024: 1, 2023: 2, 2022: 3, 2021: 4, 2020: 5, 2019: 6, 2018: 7, 2017: 8, 2016: 9, 2015: 10, 2014: 11, 2013: 12, 2012: 13, 2011: 14, 2010: 15, 2009: 16, 2008: 17, 2007: 18, 2006: 19, 2005: 20, 2004: 21, 2003: 22, 2002: 23, 2001: 24, 2000: 25, 1999: 26, 1998: 27, 1997: 28, 1996: 29, 1995: 30, 1994: 31, 1993: 32, 1992: 33, 1991: 34, 1990: 35, 1989: 36, 1988: 37, 1987: 38, 1986: 39, 1985: 40, 1984: 41, 1983: 42, 1982: 43, 1981: 44, 1980: 45, 1979: 46, 1978: 47, 1977: 48, 1976: 49, 1975: 50, 1974: 51, 1973: 52, 1972: 53, 1971: 54, 1970: 55, 1969: 56, 1968: 57, 1967: 58, 1966: 59}
{'Cardinals': 1, 'Falcons': 2, 'Ravens': 3, 'Bills': 4, 'Panthers': 5, 'Bears': 6, 'Bengals': 7, 'Browns': 8, 'Cowboys': 9, 'Broncos': 10, 'Lions': 11, 'Packers': 12, 'Texans': 13, 'Colts': 14, 'Jaguars': 15, 'Chiefs': 16, 'Raiders': 17, 'Chargers': 18, 'Rams': 19, 'Dolphins': 20, 'Vikings': 21, 'Patriots': 22, 'Saints': 23, 'Giants': 24, 'Jets': 25, 'Eagles': 26, 'Steelers': 27, '49ers': 28, 'Seahawks': 29, 'Bucca

In [21]:
insert_game = "INSERT INTO games (week, season_id, home_team_id, away_team_id, home_team_stats, away_team_stats) VALUES (%s, %s, %s, %s, %s, %s)"
insert_game_stats = """INSERT INTO game_stats (game_id, team_id, overtime, outcome, win_percentage,
    game_location, opponent_id, points_scored, points_allowed,
    passes_completed, passes_attempted, passing_yards, passing_td,
    interceptions, times_sacked, yards_sacked, yards_per_attempt,
    net_yards_per_attempt, completion_rate, passer_rating,
    rushing_attempts, rushing_yards, rushing_yards_per_attempt,
    rushing_td, field_goals_made, field_goals_attempted,
    extra_points_made, extra_points_attempted, punts, punting_yards,
    third_down_conversions, third_down_attempts,
    fourth_down_conversions, fourth_down_attempts, time_in_possession
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,%s, %s, %s, %s, %s)"""

search_previous_game = """SELECT * FROM games WHERE week = %s and season_id = %s and home_team_id = %s and away_team_id = %s"""
update_game_home_stats = """UPDATE games SET home_team_stats = %s WHERE id = %s"""
update_game_away_stats = """UPDATE games SET away_team_stats = %s WHERE id = %s"""

driver = webdriver.Chrome(service=driver_service)

for year in super_bowl_years:
    season_id = season_ids[int(year)]
    for team in teams:
        print(year, team)
        team_id = team_abr_ids[team]
        stats = get_season_stats(team, year, driver)
        for game in stats:
            opp_id = team_abr_ids[game['Opponent Abbreviation']]
            update_game_stats = update_game_home_stats

            search_previous_game_values = (game['Game Week'], season_id, opp_id, team_id)
            cursor.execute(search_previous_game, search_previous_game_values)
            game_inserted = cursor.fetchall()
            if game_inserted:
                game_id = game_inserted[0][0]
                update_game_stats = update_game_away_stats
            else:
                game_data = (game['Game Week'], season_id, team_id, opp_id, None, None)
                cursor.execute(insert_game, game_data)
                db.commit()
                game_id = cursor.lastrowid

            game_stats_data = (game_id, team_id, game['Overtime'], game['Outcome'], game['Win Percentage'], 
                game['Home Game'], opp_id, game['Points Scored'], game['Points Allowed'], 
                game['Passes Completed'], game['Passes Attempted'], game['Passing Yards'], game['Passing TD'], 
                game['Interceptions'], game['Times Sacked'], game['Yards Sacked'], game['Yards per Attempt'], 
                game['Net Yards per Attempt'], game['Completion Rate'], game['Passer Rating'], game['Rushing Attempts'], 
                game['Rushing Yards'], game['Rushing Yards per Attempt'], game['Rushing TD'], game['Field Goals Made'], 
                game['Field Goals Attempted'], game['Extra Points Made'], game['Extra Points Attempted'], game['Punts'], 
                game['Punting Yards'], game['Third Down Conversions'], game['Third Down Attempts'], 
                game['Fourth Down Conversions'], game['Fourth Down Attempts'], game['Time of Possession in Seconds'])
                
            cursor.execute(insert_game_stats, game_stats_data)
            db.commit()

            game_stats_id = cursor.lastrowid
            update_values = (game_stats_id, game_id)            
            
            cursor.execute(update_game_stats, update_values)
            db.commit()


1966 crd
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
boxscore
Game-by-game table not found.
1966 atl


KeyboardInterrupt: 

In [17]:
cursor.close()
db.close()

In [ ]:
cursor.execute("""
    SELECT COLUMN_NAME, COLUMN_TYPE
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_NAME = 'game_stats' AND TABLE_SCHEMA = 'predict_sports' AND IS_NULLABLE = 'NO'
    AND COLUMN_NAME NOT IN (
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_NAME = 'game_stats'
          AND TABLE_SCHEMA = 'predict_sports'
          AND CONSTRAINT_NAME = 'PRIMARY'
    );
""")

columns = cursor.fetchall()

# Generate and execute ALTER TABLE statements
for column in columns:
    column_name = column[0]
    column_type = column[1]
    alter_statement = f"ALTER TABLE game_stats MODIFY COLUMN {column_name} {column_type} NULL;"
    print(alter_statement)
    cursor.execute(alter_statement)

In [ ]:
insert_league = "INSERT INTO leagues (name) VALUES (%s)"
league_data = ["NFL"]

cursor.execute(insert_league, league_data)
db.commit()

print(f"{cursor.rowcount} rows inserted into leagues.")

insert_seasons = "INSERT INTO seasons (league_id, year) VALUES (%s, %s)"

for season in super_bowl_years:
    season_data = (1, int(season))
    cursor.execute(insert_seasons, season_data)
    db.commit()

print(f"{cursor.rowcount} rows inserted into leagues.")

insert_teams = "INSERT INTO teams (league_id, name, abbreviation) VALUES (%s, %s, %s)"

for key in teams:
    team_data = (1, teams[key], key)
    cursor.execute(insert_teams, team_data)
    db.commit()

print(f"{cursor.rowcount} rows inserted into leagues.")